In [1]:
import pandas as pd

# File is in the same folder as this notebook, so just the filename works
df = pd.read_excel("Telco_customer_churn.xlsx")

# 1. Shape of the data (rows, columns)
print("Shape:", df.shape)

# 2. Data types of each column
print("\nData types:\n", df.dtypes)

# 3. Missing values per column
print("\nMissing values:\n", df.isnull().sum())

# 4. Churn distribution (how many Yes vs No)

print("\nChurn value counts:\n", df['Churn Label'].value_counts())

Shape: (7043, 33)

Data types:
 CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: obje

In [2]:
import sqlite3

# Create (or connect to) a SQLite database file
conn = sqlite3.connect("churn.db")

# Push the DataFrame into a SQL table called 'customers'
df.to_sql("customers", conn, if_exists="replace", index=False)

print("Loaded into SQL. Columns available:")
print(pd.read_sql("PRAGMA table_info(customers);", conn))

Loaded into SQL. Columns available:
    cid               name     type  notnull dflt_value  pk
0     0         CustomerID     TEXT        0       None   0
1     1              Count  INTEGER        0       None   0
2     2            Country     TEXT        0       None   0
3     3              State     TEXT        0       None   0
4     4               City     TEXT        0       None   0
5     5           Zip Code  INTEGER        0       None   0
6     6           Lat Long     TEXT        0       None   0
7     7           Latitude     REAL        0       None   0
8     8          Longitude     REAL        0       None   0
9     9             Gender     TEXT        0       None   0
10   10     Senior Citizen     TEXT        0       None   0
11   11            Partner     TEXT        0       None   0
12   12         Dependents     TEXT        0       None   0
13   13      Tenure Months  INTEGER        0       None   0
14   14      Phone Service     TEXT        0       None   0
15  

In [3]:
# Churn rate by contract type
query1 = """
SELECT
    Contract,
    COUNT(*) AS customers,
    SUM(CASE WHEN "Churn Label" = 'Yes' THEN 1 ELSE 0 END) AS churned,
    ROUND(100.0 * SUM(CASE WHEN "Churn Label" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC;
"""
print("\n--- Churn rate by contract type ---")
print(pd.read_sql(query1, conn))

# Revenue (CLTV) at risk by segment
query2 = """
SELECT
    Contract,
    "Internet Service",
    COUNT(*) AS at_risk_customers,
    ROUND(SUM(CLTV), 2) AS cltv_at_risk,
    ROUND(SUM("Monthly Charges"), 2) AS monthly_revenue_at_risk
FROM customers
WHERE "Churn Label" = 'Yes'
GROUP BY Contract, "Internet Service"
ORDER BY monthly_revenue_at_risk DESC
LIMIT 5;
"""
print("\n--- Revenue at risk (top 5 segments) ---")
print(pd.read_sql(query2, conn))


--- Churn rate by contract type ---
         Contract  customers  churned  churn_rate_pct
0  Month-to-month       3875     1655            42.7
1        One year       1473      166            11.3
2        Two year       1695       48             2.8

--- Revenue at risk (top 5 segments) ---
         Contract Internet Service  at_risk_customers  cltv_at_risk  \
0  Month-to-month      Fiber optic               1162     4734281.0   
1  Month-to-month              DSL                394     1603091.0   
2        One year      Fiber optic                104      494570.0   
3        One year              DSL                 53      222810.0   
4        Two year      Fiber optic                 31      156454.0   

   monthly_revenue_at_risk  
0                100482.00  
1                 18367.25  
2                 10571.95  
3                  3356.25  
4                  3246.10  


In [4]:
# 1. Total Charges is stored as text — convert to numeric
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

# 2. Check how many rows became NaN after conversion (these are the blank strings)
print("Rows with missing Total Charges after conversion:", df["Total Charges"].isnull().sum())

# 3. Look at those rows - what do they have in common?
print(df[df["Total Charges"].isnull()][["CustomerID", "Tenure Months", "Monthly Charges", "Total Charges"]])

# 4. Fix it: these are new customers (tenure = 0), so Total Charges should just equal Monthly Charges (or 0)
df["Total Charges"] = df["Total Charges"].fillna(df["Monthly Charges"])

# 5. Confirm no more missing values
print("\nAfter fix, missing Total Charges:", df["Total Charges"].isnull().sum())

# 6. Create the numeric churn flag we'll use as the model's target
df["Churn Flag"] = (df["Churn Label"] == "Yes").astype(int)
print("\nChurn Flag distribution:\n", df["Churn Flag"].value_counts())

Rows with missing Total Charges after conversion: 11
      CustomerID  Tenure Months  Monthly Charges  Total Charges
2234  4472-LVYGI              0            52.55            NaN
2438  3115-CZMZD              0            20.25            NaN
2568  5709-LVOEQ              0            80.85            NaN
2667  4367-NUYAO              0            25.75            NaN
2856  1371-DWPAZ              0            56.05            NaN
4331  7644-OMVMY              0            19.85            NaN
4687  3213-VVOLG              0            25.35            NaN
5104  2520-SGTTA              0            20.00            NaN
5719  2923-ARZLG              0            19.70            NaN
6772  4075-WKNIU              0            73.35            NaN
6840  2775-SEFEE              0            61.90            NaN

After fix, missing Total Charges: 0

Churn Flag distribution:
 Churn Flag
0    5174
1    1869
Name: count, dtype: int64


In [5]:
# Columns we'll feed into the model
numeric_cols = ["Tenure Months", "Monthly Charges", "Total Charges"]

categorical_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method"
]

X = df[numeric_cols + categorical_cols]
y = df["Churn Flag"]

print("Feature matrix shape:", X.shape)
print("\nCategorical column unique values:")
for col in categorical_cols:
    print(f"  {col}: {df[col].unique()}")

Feature matrix shape: (7043, 19)

Categorical column unique values:
  Gender: ['Male' 'Female']
  Senior Citizen: ['No' 'Yes']
  Partner: ['No' 'Yes']
  Dependents: ['No' 'Yes']
  Phone Service: ['Yes' 'No']
  Multiple Lines: ['No' 'Yes' 'No phone service']
  Internet Service: ['DSL' 'Fiber optic' 'No']
  Online Security: ['Yes' 'No' 'No internet service']
  Online Backup: ['Yes' 'No' 'No internet service']
  Device Protection: ['No' 'Yes' 'No internet service']
  Tech Support: ['No' 'Yes' 'No internet service']
  Streaming TV: ['No' 'Yes' 'No internet service']
  Streaming Movies: ['No' 'Yes' 'No internet service']
  Contract: ['Month-to-month' 'Two year' 'One year']
  Paperless Billing: ['Yes' 'No']
  Payment Method: ['Mailed check' 'Electronic check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

# Split BEFORE fitting anything, so the test set stays truly unseen
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessing: scale numeric columns, one-hot encode categorical columns
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

# Model
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
)

# Full pipeline: preprocessing + model in one object
pipe = Pipeline([("prep", preprocess), ("model", model)])

# Train
pipe.fit(X_train, y_train)

# Predict probabilities on the held-out test set
y_prob_test = pipe.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= 0.5).astype(int)

# Evaluate
auc = roc_auc_score(y_test, y_prob_test)
print("Test ROC-AUC:", round(auc, 4))
print("\nClassification report:\n", classification_report(y_test, y_pred_test))

Test ROC-AUC: 0.8539

Classification report:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [7]:
import shap
import numpy as np

# Transform the raw features the same way the model saw them during training
X_test_transformed = pipe.named_steps["prep"].transform(X_test)
feature_names = pipe.named_steps["prep"].get_feature_names_out()

# Explain using the trained XGBoost model directly (fast TreeExplainer)
explainer = shap.TreeExplainer(pipe.named_steps["model"])
shap_values = explainer.shap_values(X_test_transformed)

# Global importance: which features matter most across ALL customers?
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

print("Top 10 most important features overall:\n")
print(importance_df.head(10).to_string(index=False))

C:\Users\Admin\anaconda3\envs\dev_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Top 10 most important features overall:

                             feature  mean_abs_shap
        cat__Contract_Month-to-month       0.765771
                  num__Tenure Months       0.516139
                  cat__Dependents_No       0.477708
                num__Monthly Charges       0.248146
   cat__Internet Service_Fiber optic       0.222408
             cat__Online Security_No       0.216368
                  num__Total Charges       0.193200
                cat__Tech Support_No       0.166544
cat__Payment Method_Electronic check       0.164399
              cat__Contract_Two year       0.140273


In [18]:

X_full_transformed = pipe.named_steps["prep"].transform(X)
full_prob = pipe.predict_proba(X)[:, 1]

df["Churn Probability"] = full_prob

def segment(prob):
    if prob >= 0.6:
        return "High Risk"
    elif prob >= 0.3:
        return "Medium Risk"
    else:
        return "Low Risk"

df["Risk Segment"] = df["Churn Probability"].apply(segment)

# Per-customer SHAP: find each customer's single biggest risk-increasing feature
full_shap_values = explainer.shap_values(X_full_transformed)
top_idx = np.argmax(full_shap_values, axis=1)
top_reason_raw = feature_names[top_idx]

def prettify(feat):
    feat = feat.replace("num__", "").replace("cat__", "")
    return feat.replace("_", " ")

df["Top Churn Driver"] = [prettify(f) for f in top_reason_raw]

# Summary: how many customers land in each segment, and their revenue
summary = df.groupby("Risk Segment").agg(
    customers=("CustomerID", "count"),
    avg_monthly_charge=("Monthly Charges", "mean"),
    total_cltv=("CLTV", "sum"),
).round(2)

print(summary)
print("\nSample of highest-risk customers:")
print(df.sort_values("Churn Probability", ascending=False)
        [["CustomerID", "Tenure Months", "Contract", "Churn Probability", "Risk Segment", "Top Churn Driver"]]
        .head(10).to_string(index=False))

              customers  avg_monthly_charge  total_cltv
Risk Segment                                           
High Risk          1126               80.33     4503679
Low Risk           4430               58.40    20174205
Medium Risk        1487               71.94     6313399

Sample of highest-risk customers:
CustomerID  Tenure Months       Contract  Churn Probability Risk Segment Top Churn Driver
2012-NWRPA             11 Month-to-month           0.971052    High Risk  Monthly Charges
1455-UGQVH             10 Month-to-month           0.967529    High Risk  Monthly Charges
0897-FEGMU             11 Month-to-month           0.965116    High Risk  Monthly Charges
0107-YHINA              1 Month-to-month           0.964406    High Risk    Tenure Months
7216-EWTRS              1 Month-to-month           0.962034    High Risk    Tenure Months
2454-RPBRZ             12 Month-to-month           0.957411    High Risk  Monthly Charges
1450-GALXR             10 Month-to-month           0.94

In [20]:
# Full customer-level file — powers your drilldown table
export_cols = [
    "CustomerID", "Tenure Months", "Monthly Charges", "Total Charges", "CLTV",
    "Contract", "Internet Service", "Payment Method",
    "Churn Label", "Churn Reason", "Churn Probability", "Risk Segment", "Top Churn Driver"
]
df[export_cols].sort_values("Churn Probability", ascending=False).to_csv("scored_customers.csv", index=False)

# Segment summary — powers your KPI cards and top-level charts
summary_export = df.groupby("Risk Segment").agg(
    customers=("CustomerID", "count"),
    avg_monthly_charge=("Monthly Charges", "mean"),
    total_cltv=("CLTV", "sum"),
).round(2).reset_index()
summary_export.to_csv("powerbi_segment_summary.csv", index=False)

print("Exported scored_customers.csv and powerbi_segment_summary.csv")
print("\nFiles in folder now:")
import os
print([f for f in os.listdir() if f.endswith(".csv")])

Exported scored_customers.csv and powerbi_segment_summary.csv

Files in folder now:
['powerbi_segment_summary.csv', 'scored_customers.csv']
